This Python script is part of a data science project aimed at analyzing motorcycle specifications and market trends using data extracted from Bikez.com via web scraping.

The primary goal of this script is to clean, validate, and transform the raw scraped data into a structured and reliable dataset, ensuring its suitability for downstream tasks such as exploratory analysis, statistical modeling, or machine learning.

In [1]:
import pandas as pd
import numpy as np
import os
import requests
from bs4 import BeautifulSoup
import re

In [2]:
# Read CSV file and save it to a dataframe:
df = pd.read_csv("Bikez-All-Years.csv ")

# List of column names:
df

,Motorcycle name,Model year,Category,Price as new,Engine size,Type of engine,Power output,Torque,Transmission type,Clutch,...,Rear tire,Front brakes,Rear brakes,"Weight incl. oil, gas, etc",Dry weight,Seat height,Overall height,Overall length,Fuel capacity,Oil capacity
0,Hildebrand-Wolfmüller Motorrad,1894,Allround,NaN,1489.0 ccm (90.86 cubic inches),"Twin, four-stroke",2.5HP(1.8kW)) @ 240RPM,NaN,Shaft drive (cardan) (final drive),NaN,...,NaN,NaN,NaN,NaN,50.0 kg (110.2 pounds),NaN,NaN,NaN,NaN,NaN
1,Hildebrand-Wolfmüller Motorrad,1895,Allround,NaN,1489.0 ccm (90.86 cubic inches),"Twin, four-stroke",2.5HP(1.8kW)) @ 240RPM,NaN,Shaft drive (cardan) (final drive),NaN,...,NaN,NaN,NaN,NaN,50.0 kg (110.2 pounds),NaN,NaN,NaN,NaN,NaN
2,Millet Motorcycle,1895,Allround,NaN,625.0 ccm (38.14 cubic inches),Radial,1.2HP(0.9kW)) @ 180RPM,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Excelsior Motor Bicycle,1896,Allround,NaN,234.0 ccm (14.28 cubic inches),"Single cylinder, four-stroke",1.3HP(0.9kW)),NaN,Chain (final drive),NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Hildebrand-Wolfmüller Motorrad,1896,Allround,NaN,1489.0 ccm (90.86 cubic inches),"Twin, four-stroke",2.5HP(1.8kW)) @ 240RPM,NaN,Shaft drive (cardan) (final drive),NaN,...,NaN,NaN,NaN,NaN,50.0 kg (110.2 pounds),NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
42399,Znen Tank II,2025,Scooter,NaN,150.0 ccm (9.15 cubic inches),"Single cylinder, four-stroke",10.2HP(7.4kW)) @ 7500RPM,10.1 Nm (1.0 kgf-m or 7.4 ft.lbs) @ 6000RPM,Belt (final drive),NaN,...,120/70-12,Single disc,Singledisc,125.0 kg (275.6pounds),116.0 kg(255.7 pounds),"760 mm (29.9 inches)If adjustable, lowest sett...",1220 mm (48.0inches),1900 mm(74.8 inches),7.40 litres (1.96 USgallons),NaN
42400,Znen Veracruz,2025,Scooter,NaN,50.0 ccm (3.05cubic inches),"Single cylinder, four-stroke",3.2HP(2.4kW)) @ 8000RPM,2.8 Nm (0.3 kgf-m or 2.1 ft.lbs) @ 6500RPM,Belt (final drive),NaN,...,NaN,Single disc,Single disc,96.0 kg (211.6 pounds),90.0 kg (198.4 pounds),"780 mm (30.7 inches) If adjustable, lowest set...",1135 mm (44.7 inches),1900 mm (74.8 inches),6.20 litres(1.64 US gallons),NaN
42401,Zontes 368G,2025,Scooter,NaN,368.0 ccm (22.46 cubic inches),"Single cylinder,four-stroke",38.2HP(27.9kW)) @ 7500RPM,40.0 Nm (4.1 kgf-m or 29.5 ft.lbs) @ 6000RPM,NaN,Wettype multi-pieces,...,150/70-14,Singledisc.ABS,Single disc. ABS,198.0 kg (436.5 pounds),NaN,"795 mm (31.3 inches) If adjustable, lowest set...",1370 mm(53.9inches),2230 mm (87.8 inches),17.50 litres (4.62 USgallons),NaN
42402,Zontes 703F Advendture,2025,Super motard,NaN,699.0 ccm (42.65 cubic inches),"In-line three, four-stroke",95.9HP(70.0kW)) @ 10000RPM,76.0 Nm (7.7 kgf-m or 56.1 ft.lbs) @ 7500RPM,Chain (final drive),Wetmulti-disk,...,170/60-R17,Singledisc. ABS,Single disc. ABS,241.0 kg (531.3 pounds),NaN,"825 mm (32.5 inches) If adjustable, lowest set...",1395 mm (54.9inches),2065 mm(81.3 inches),22.00 litres (5.81 US gallons),NaN


In [3]:
# Each column will be processed individually, with data formatted appropriately for later analysis.

# Motorcycle name: to be splitted into actual motorbike name and its brand.
# Problem: some brands have more than a word for their name so that 
# has to be handled to split the columns properly.

# Save list of brands from Bikez.com and use it for data cleaning:
r = requests.get("https://bikez.com/brands/index.php")
soup = BeautifulSoup(r.text, "html.parser")

table = soup.find("table", class_="zebra")
brand_links = table.find_all("a", href=re.compile("_models.php"))

# Brand names appear as "brand + motorcycles" on Bikez.com:
brands = [link.text.replace(" motorcycles", "") for link in brand_links]

# List of brands with more than 2 words to handle data cleaning:
multiword_brands = [brand for brand in brands if " " in brand]

# Function to split "Motorcycle name" into "Brand" and "Model":
def split_brand_model(motorcycle_name):
    # Step 1: Multiword brands:
    for brand in multiword_brands:
        if motorcycle_name.startswith(brand):
            model = motorcycle_name[len(brand):].strip() # Starts right after brand
            return brand, model
    
    # Step 2: Simple brands:
    parts = motorcycle_name.split(" ", 1)  # Divides string only after first blankspace
    
    brand = parts[0]  # Brand is first word
    model = parts[1] if len(parts) > 1 else ""  # Model is rest
 
    return brand, model

# Finally apply to dataframe and re-order columns:
df[["Brand", "Model"]] = df["Motorcycle name"].apply(lambda x: pd.Series(split_brand_model(x)))

df.drop("Motorcycle name", axis=1, inplace=True)

column_order = ["Brand", "Model"] + [col for col in df.columns if col not in ["Brand", "Model"]]
df = df[column_order]

df.head(1)

,Brand,Model,Model year,Category,Price as new,Engine size,Type of engine,Power output,Torque,Transmission type,...,Rear tire,Front brakes,Rear brakes,"Weight incl. oil, gas, etc",Dry weight,Seat height,Overall height,Overall length,Fuel capacity,Oil capacity
0,Hildebrand-Wolfmüller,Motorrad,1894,Allround,NaN,1489.0 ccm (90.86 cubic inches),"Twin, four-stroke",2.5HP(1.8kW)) @ 240RPM,NaN,Shaft drive (cardan) (final drive),...,NaN,NaN,NaN,NaN,50.0 kg (110.2 pounds),NaN,NaN,NaN,NaN,NaN


In [4]:
# Column "Model year": No cleaning required (values already consistent).

# Category:
print(df["Category"].unique())

['Allround' 'Sport' 'Custom / cruiser' 'Naked bike' 'Enduro / offroad'
 'Classic' 'Cross / motocross' 'Trial' 'Scooter' 'Sport touring' 'Touring'
 'Speedway' 'Prototype / concept model' 'Minibike, sport'
 'Minibike, cross' 'Super motard' 'Unspecified category' 'ATV' 'Nakedbike'
 'Custom /cruiser' 'Enduro /offroad' 'Enduro/ offroad' 'Cross/ motocross'
 'Sporttouring' 'Supermotard' 'Minibike,sport' 'Custom/ cruiser'
 'Cross /motocross' 'Enduro/offroad' 'Custom/cruiser' 'Cross/motocross'
 'Minibike,cross']


In [5]:
# The column contains minor naming variations that will be standardized using 
# a regex-based normalization dictionary, resulting in a consolidated list of categories.

normalization_rules = {
        r"naked\s*bikes?": "Naked bike",
        r"prototype\s*[/\\]?\s*concept\s*model": "Prototype",
        r"super\s*motards?": "Super motard",
        r"sport\s*tourings?": "Sport touring",
        r"enduro\s*[/\\]?\s*off\s*roads?": "Enduro/offroad",
        r"cross\s*[/\\]?\s*moto\s*cross": "Cross/motocross",
        r"custom\s*[/\\]?\s*cruisers?": "Custom/cruiser",
        r"minibike\s*[,/\\]?\s*sport": "Minibike/sport",
        r"minibike\s*[,/\\]?\s*cross": "Minibike/cross"
    }    

# Apply rules to "Category" column":
for pattern, replacement in normalization_rules.items():
    df["Category"] = df["Category"].str.replace(
        pattern, 
        replacement, 
        case=False,  # Ignore lower/upper case
        regex=True
    )

# Category (clean data):
print(df["Category"].unique())

['Allround' 'Sport' 'Custom/cruiser' 'Naked bike' 'Enduro/offroad'
 'Classic' 'Cross/motocross' 'Trial' 'Scooter' 'Sport touring' 'Touring'
 'Speedway' 'Prototype' 'Minibike/sport' 'Minibike/cross' 'Super motard'
 'Unspecified category' 'ATV']


In [6]:
# Price as new:
# This field is critical for the analysis. The available sample of 
# ~10,000 motorcycles with documented prices should provide sufficient 
# data volume for training the price estimation model.

print(f"Amount of bikes with registered price: ", df['Price as new'].notna().sum())
print(f"Amount of bikes without registered price: ", df['Price as new'].isna().sum())

Amount of bikes with registered price:  9581
Amount of bikes without registered price:  32823


In [7]:
# A substantial sample of ~10,000 motorcycles have documented prices, which should provide 
# sufficient data volume for training our price estimation machine learning model.
# Rename it to "Price as new (USD)"" and store only the numerical value in the column:

df["Price as new"] = df["Price as new"].dropna().str.split(" ").str[1].str.replace(".", "")
df = df.rename(columns={"Price as new": "Price as new (USD)"})

df[df["Price as new (USD)"].notna()].head(1)

,Brand,Model,Model year,Category,Price as new (USD),Engine size,Type of engine,Power output,Torque,Transmission type,...,Rear tire,Front brakes,Rear brakes,"Weight incl. oil, gas, etc",Dry weight,Seat height,Overall height,Overall length,Fuel capacity,Oil capacity
1014,Harley-Davidson,Model FL,1948,Touring,650,1212.0 ccm (73.96 cubic inches),"V2, four-stroke",50.0HP(36.5kW)) @ 4800RPM,NaN,Chain (final drive),...,5.00-16,Expanding brake (drum brake),Expanding brake (drum brake),NaN,256.0 kg (564.4 pounds),NaN,NaN,NaN,14.19 litres (3.75 US gallons),3.80 litres (4.02 US quarts)


In [8]:
# Engine size:
print(f"Amount of bikes with engine size: ", df["Engine size"].notna().sum())
print(f"Amount of bikes without engine size: ", df["Engine size"].isna().sum())

Amount of bikes with engine size:  41079
Amount of bikes without engine size:  1325


In [9]:
# Convert the Engine size column to numeric values 
# representing cubic centimeters and rename it for clarity:

df["Engine size"] = df["Engine size"].str.split(".").str[0]
df = df.rename(columns={"Engine size": "Engine size (cubic centimetres)"})

df[df["Engine size (cubic centimetres)"].notna()].head(1)

,Brand,Model,Model year,Category,Price as new (USD),Engine size (cubic centimetres),Type of engine,Power output,Torque,Transmission type,...,Rear tire,Front brakes,Rear brakes,"Weight incl. oil, gas, etc",Dry weight,Seat height,Overall height,Overall length,Fuel capacity,Oil capacity
0,Hildebrand-Wolfmüller,Motorrad,1894,Allround,NaN,1489,"Twin, four-stroke",2.5HP(1.8kW)) @ 240RPM,NaN,Shaft drive (cardan) (final drive),...,NaN,NaN,NaN,NaN,50.0 kg (110.2 pounds),NaN,NaN,NaN,NaN,NaN


In [10]:
# Type of engine:
df["Type of engine"].unique()

array(['Twin, four-stroke', 'Radial', 'Single cylinder, four-stroke',
       'In-line three, four-stroke', 'In-line three, two-stroke',
       'Single cylinder, two-stroke', 'In-line four, two-stroke',
       'V2, four-stroke', 'V2, two-stroke',
       'Two cylinder boxer, four-stroke', 'Twin, two-stroke',
       'In-line four, four-stroke', 'Square four cylinder',
       'V4, four-stroke', 'Four cylinder boxer, four-stroke',
       'Two cylinder boxer, two-stroke', 'In-line six, four-stroke',
       'V8, four-stroke', 'Single disk Wankel', 'Dual disk Wankel',
       'Four cylinder boxer, two-stroke', 'V4, two-stroke',
       'V3, two-stroke', 'Six cylinder boxer, four-stroke',
       'In-line six, two-stroke', 'Diesel', nan, 'Gas turbine',
       'V10, four-stroke', 'Electric', 'V6, four-stroke',
       'Single cylinder,four-stroke', 'Twin,four-stroke',
       'V4,four-stroke', 'Singlecylinder, four-stroke',
       'Single cylinder,two-stroke', 'In-linefour, four-stroke',
       'In-l

In [11]:
# This column contains variations that don't provide additional analytical value, 
# so it will remain unprocessed.

In [12]:
# Power output:

# Split the Power output column into two numeric columns: horsepower (HP) 
# and the RPM at which this power is achieved.
df[["Power output (HP)", "Power output (@RPM)"]] = df["Power output"].str.split("@", n=1, expand=True)
df["Power output (HP)"] = df["Power output (HP)"].str.split("HP").str[0].astype(float)
df["Power output (@RPM)"] = df["Power output (@RPM)"].str.split("RPM").str[0].astype(float)

# Place new columns right before "Torque", where old "Power output" column was:
torque_pos = df.columns.get_loc("Torque")

# Drop old Power column:
df.drop("Power output", axis=1, inplace=True)

# New column order:
column_order = [col for col in df.columns if col not in ["Power output (HP)", "Power output (@RPM)"]]  # Exclude new columns
column_order.insert(torque_pos - 1, "Power output (HP)")  # Insert new columns where the old column was
column_order.insert(torque_pos, "Power output (@RPM)")

# Reorder dataframe:
df = df[column_order]

df.columns

Index(['Brand', 'Model', 'Model year', 'Category', 'Price as new (USD)',
       'Engine size (cubic centimetres)', 'Type of engine',
       'Power output (HP)', 'Power output (@RPM)', 'Torque',
       'Transmission type', 'Clutch', 'Fuel consumption', 'Front tire',
       'Rear tire', 'Front brakes', 'Rear brakes',
       'Weight incl. oil, gas, etc', 'Dry weight', 'Seat height',
       'Overall height', 'Overall length', 'Fuel capacity', 'Oil capacity'],
      dtype='object')

In [13]:
# Torque:
# Split into Torque (Nm) and Torque (@RPM):
df[["Torque (Nm)", "Torque (@RPM)"]] = df["Torque"].str.split("@", n=1, expand=True)
df["Torque (Nm)"] = df["Torque (Nm)"].str.split("Nm").str[0].astype(float)
df["Torque (@RPM)"] = df["Torque (@RPM)"].str.split("RPM").str[0].astype(float)

# Place new columns right before "Tranmission Type", where old "Torque" column was:
transmission_pos = df.columns.get_loc("Transmission type")

# Drop old Torque column:
df.drop("Torque", axis=1, inplace=True)

# New column order:
column_order = [col for col in df.columns if col not in ["Torque (Nm)", "Torque (@RPM)"]]  # Exclude new columns
column_order.insert(transmission_pos - 1, "Torque (Nm)")  # Insert new columns where the old column was
column_order.insert(transmission_pos, "Torque (@RPM)")

# Reorder dataframe:
df = df[column_order]

df.columns

Index(['Brand', 'Model', 'Model year', 'Category', 'Price as new (USD)',
       'Engine size (cubic centimetres)', 'Type of engine',
       'Power output (HP)', 'Power output (@RPM)', 'Torque (Nm)',
       'Torque (@RPM)', 'Transmission type', 'Clutch', 'Fuel consumption',
       'Front tire', 'Rear tire', 'Front brakes', 'Rear brakes',
       'Weight incl. oil, gas, etc', 'Dry weight', 'Seat height',
       'Overall height', 'Overall length', 'Fuel capacity', 'Oil capacity'],
      dtype='object')

In [14]:
# Transmission type:
# Unify all transmission types with a Regex dictionary:

df["Transmission type"].unique()

array(['Shaft drive (cardan)  (final drive)', nan, 'Chain  (final drive)',
       'Belt  (final drive)', 'Chain  (finaldrive)', 'Belt  (finaldrive)',
       'Chain(final drive)', 'Chain(finaldrive)', 'Belt(final drive)',
       'Shaft drive (cardan)(final drive)',
       'Shaft drive (cardan)  (finaldrive)',
       'Shaftdrive (cardan)  (final drive)',
       'Shaft drive (cardan)(finaldrive)',
       'Shaftdrive (cardan)(final drive)',
       'Shaft drive(cardan)  (final drive)',
       'Shaft drive(cardan)(final drive)', 'Belt(finaldrive)',
       'Shaftdrive(cardan)  (final drive)',
       'Shaft drive(cardan)  (finaldrive)'], dtype=object)

In [15]:
normalization_rules2 = {
        r"Shaft\s*drive\s*\(?\s*cardan\s*\)?\s*\(?\s*final\s*drive\s*\)?": "Shaft drive (cardan) (final drive)",
        r"Chain\s*\(?\s*final\s*drive\s*\)?": "Chain (final drive)",
        r"Belt\s*\(?\s*final\s*drive\s*\)?": "Belt (final drive)"
    }    

# Apply rules to "Transmission type" column:
for pattern, replacement in normalization_rules2.items():
    df["Transmission type"] = df["Transmission type"].str.replace(
        pattern, 
        replacement, 
        case=False,  # Ignore lower/upper case
        regex=True
    )

df["Transmission type"].unique()

array(['Shaft drive (cardan) (final drive)', nan, 'Chain (final drive)',
       'Belt (final drive)'], dtype=object)

In [16]:
# Clutch: this column has high cardinality (making it hard to be cleaned easily)
# without providing significant analytical value for the current 
# project goals, so it will remain as-is.

In [17]:
# Standardize fuel consumption measurements to L/100km by extracting numeric values::
df["Fuel consumption"] = df["Fuel consumption"].str.split("litres").str[0]
df = df.rename(columns={"Fuel consumption": "Fuel consumption (L/100km)"})


In [18]:
# Columns "Tires" and "Brakes" retain original values: 
# No standardization needed for current analysis.

In [19]:
# Keep both weights in kilograms (numeric values only).
df["Weight incl. oil, gas, etc"] = df["Weight incl. oil, gas, etc"].str.split("kg").str[0]
df["Dry weight"] = df["Dry weight"] .str.split("kg").str[0]

df = df.rename(columns={"Weight incl. oil, gas, etc": "Weight incl. oil, gas, etc (kg)"})
df = df.rename(columns={"Dry weight": "Dry weight (kg)"})

In [20]:
# Seat height.
# Overall height.
# Overall length.

# Use millimetres as the unit for this columns (numbers only, no "mm" suffix).
df["Seat height"] = df["Seat height"].str.split("mm").str[0]
df = df.rename(columns={"Seat height": "Seat height (mm)"})
df["Overall height"] = df["Overall height"].str.split("mm").str[0]
df = df.rename(columns={"Overall height": "Overall height (mm)"})
df["Overall length"] = df["Overall length"].str.split("mm").str[0]
df = df.rename(columns={"Overall length": "Overall length (mm)"})

df[df[["Seat height (mm)", "Overall height (mm)", "Overall length (mm)"]].notna().all(axis=1)].head(1)


,Brand,Model,Model year,Category,Price as new (USD),Engine size (cubic centimetres),Type of engine,Power output (HP),Power output (@RPM),Torque (Nm),...,Rear tire,Front brakes,Rear brakes,"Weight incl. oil, gas, etc (kg)",Dry weight (kg),Seat height (mm),Overall height (mm),Overall length (mm),Fuel capacity,Oil capacity
444,NSU,301T,1929,Sport,NaN,298,"Single cylinder, four-stroke",7.0,4000.0,NaN,...,3-25,Expanding brake (drum brake),Expanding brake (drum brake),NaN,130.0,690,920,1990,10.00 litres (2.64 US gallons),NaN


In [21]:
# Fuel capacity:
# Oil capacity:

# Leave as amount of litres:
df["Fuel capacity"] = df["Fuel capacity"].str.split("litres").str[0]
df = df.rename(columns={"Fuel capacity": "Fuel capacity (litres)"})
df["Oil capacity"] = df["Oil capacity"].str.split("litres").str[0]
df = df.rename(columns={"Oil capacity": "Oil capacity (litres)"})

df[df[["Fuel capacity (litres)", "Oil capacity (litres)"]].notna().all(axis=1)].head(1)

,Brand,Model,Model year,Category,Price as new (USD),Engine size (cubic centimetres),Type of engine,Power output (HP),Power output (@RPM),Torque (Nm),...,Rear tire,Front brakes,Rear brakes,"Weight incl. oil, gas, etc (kg)",Dry weight (kg),Seat height (mm),Overall height (mm),Overall length (mm),Fuel capacity (litres),Oil capacity (litres)
179,Harley-Davidson,Sport Twin,1919,Sport,NaN,590,"Twin, four-stroke",NaN,NaN,NaN,...,3-26,NaN,NaN,NaN,120.0,NaN,NaN,NaN,10.00,1.90


In [22]:
# Export the cleaned dataset to a CSV file to be used in subsequent project phases. 
# The output file contains standardized, analysis-ready fields.

df.to_csv('Bikez-All-Years-Clean.csv')